# Notebook 2: Message Passing & Graph Convolutional Networks

This is the conceptual heart of GNNs. We'll go from "why can't CNNs work on graphs?" all the way to implementing a GCN layer ourselves in raw PyTorch, then switch to PyG.

**Prerequisites:** Notebook 1 (graph representations, PyG Data object)

**What you'll learn:**
- The message passing framework
- GCN derivation from spectral graph theory (intuitive version)
- Implementing GCN *from scratch* with PyTorch matrix ops
- Using PyG's `GCNConv` layer
- Training a GCN for node classification on the Karate Club graph

**By the end:** you'll implement your own GCN layer from scratch and train it.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_networkx

torch.manual_seed(42)
np.random.seed(42)

---
## 1. Why CNNs Fail on Graphs

CNNs work by sliding a fixed-size kernel over a regular grid. Each pixel has exactly 8 neighbors (or 4 on boundaries), always in the same spatial arrangement. This regularity is what makes weight sharing possible.

Graphs break this assumption:
- Node 0 might have 2 neighbors, node 1 might have 47
- There's no canonical ordering of neighbors (no "left", "right", "up")
- The same edge pattern can look geometrically different depending on layout

**The fix:** instead of fixed kernels, we aggregate over a node's *neighborhood* — regardless of size or order. This is message passing.

---
## 2. The Message Passing Framework

Every GNN variant can be written as:

```
for each node v:
    m_v = AGGREGATE({h_u : u in N(v)})    # collect messages from neighbors
    h_v_new = UPDATE(h_v, m_v)            # update own representation
```

Where:
- `h_v` = current embedding/feature of node v
- `N(v)` = neighbors of v
- `AGGREGATE` = some permutation-invariant function (sum, mean, max, LSTM...)
- `UPDATE` = some function (linear layer, MLP...)

Different GNN architectures = different choices of AGGREGATE and UPDATE. Stack multiple rounds = each node sees information from further away (k rounds → k-hop neighborhood).

In [ ]:
# Visualize what '1-hop' vs '2-hop' neighborhoods mean
G = nx.karate_club_graph()
target_node = 0

hop1 = set(G.neighbors(target_node))
hop2 = set()
for n in hop1:
    hop2.update(G.neighbors(n))
hop2 -= hop1
hop2.discard(target_node)

color_map = []
for node in G.nodes():
    if node == target_node:
        color_map.append('red')
    elif node in hop1:
        color_map.append('orange')
    elif node in hop2:
        color_map.append('gold')
    else:
        color_map.append('lightgray')

fig, ax = plt.subplots(figsize=(9, 6))
pos = nx.spring_layout(G, seed=7)
nx.draw(G, pos, ax=ax, node_color=color_map, node_size=400,
        edge_color='lightgray', with_labels=True, font_size=7)

from matplotlib.patches import Patch
legend = [Patch(color='red', label=f'Target (node {target_node})'),
          Patch(color='orange', label='1-hop neighborhood'),
          Patch(color='gold', label='2-hop neighborhood'),
          Patch(color='lightgray', label='Others')]
ax.legend(handles=legend)
ax.set_title('Message passing: after 2 GNN layers, node 0 has info from orange+gold nodes')
plt.tight_layout()
plt.show()

print(f'Node {target_node} degree: {len(hop1)}')
print(f'1-hop size: {len(hop1)}')
print(f'2-hop size: {len(hop2)}')

---
## 3. Graph Convolutional Networks (GCN)

**Paper:** Kipf & Welling, 2017 — "Semi-Supervised Classification with Graph Convolutional Networks"

The GCN layer formula is:

$$H^{(l+1)} = \sigma\left(\tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2} H^{(l)} W^{(l)}\right)$$

Where:
- $\tilde{A} = A + I$ — adjacency matrix **with self-loops added**
- $\tilde{D}$ — degree matrix of $\tilde{A}$
- $H^{(l)}$ — node features at layer $l$ (shape: N × F)
- $W^{(l)}$ — learnable weight matrix (shape: F_in × F_out)
- $\sigma$ — non-linearity (ReLU)

**Intuition:** For each node, sum up features of all neighbors (including itself), normalize by degree to prevent scale explosion, then multiply by a learnable weight matrix.

**The normalization trick:** Without $D^{-1/2} A D^{-1/2}$, high-degree nodes would dominate. Symmetric normalization makes aggregation degree-agnostic.

### Step-by-step:
1. Add self-loops: $\tilde{A} = A + I$
2. Compute normalized adjacency: $\hat{A} = \tilde{D}^{-1/2} \tilde{A} \tilde{D}^{-1/2}$
3. Linear transform: $H W$
4. Aggregate: $\hat{A} (H W)$
5. Apply non-linearity

In [ ]:
# ============================================================
# GCN Layer from scratch — pure PyTorch + NumPy
# ============================================================

class GCNLayerScratch(nn.Module):
    """Single GCN layer using explicit adjacency matrix operations."""
    
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.W = nn.Linear(in_features, out_features, bias=False)
        nn.init.xavier_uniform_(self.W.weight)
    
    def forward(self, X: torch.Tensor, A_hat: torch.Tensor) -> torch.Tensor:
        # A_hat: pre-computed normalized adjacency (N x N)
        # X:     node features (N x in_features)
        support = self.W(X)          # (N x out_features)
        out = A_hat @ support        # aggregate
        return out


def compute_normalized_adjacency(A: np.ndarray) -> torch.Tensor:
    """Compute D^{-1/2} (A + I) D^{-1/2}."""
    N = A.shape[0]
    A_tilde = A + np.eye(N)                          # add self-loops
    D_tilde = np.diag(A_tilde.sum(axis=1))           # degree matrix
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D_tilde)))
    A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt         # symmetric normalization
    return torch.tensor(A_hat, dtype=torch.float)


# Quick forward pass test
torch.manual_seed(0)
A_demo = np.array([[0,1,1,0],[1,0,1,0],[1,1,0,1],[0,0,1,0]], dtype=float)
A_hat_demo = compute_normalized_adjacency(A_demo)
X_demo = torch.randn(4, 3)   # 4 nodes, 3 features

layer = GCNLayerScratch(3, 2)
out = layer(X_demo, A_hat_demo)
print(f'Input X:  {X_demo.shape}')
print(f'A_hat:    {A_hat_demo.shape}')
print(f'Output:   {out.shape}')
print(f'\nOutput (first pass, no training):\n{out.detach().numpy().round(3)}')

In [ ]:
# ============================================================
# 2-layer GCN using scratch layers
# ============================================================

class GCNScratch(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.conv1 = GCNLayerScratch(in_features, hidden)
        self.conv2 = GCNLayerScratch(hidden, out_features)
    
    def forward(self, X, A_hat):
        h = F.relu(self.conv1(X, A_hat))
        h = F.dropout(h, p=0.5, training=self.training)
        h = self.conv2(h, A_hat)
        return h  # raw logits — apply softmax/log_softmax separately

# Load Karate Club
dataset = KarateClub()
data = dataset[0]

# Build adjacency from edge_index
N = data.num_nodes
A = np.zeros((N, N))
for i, j in data.edge_index.T.numpy():
    A[i, j] = 1.0
A_hat = compute_normalized_adjacency(A)

print(f'Graph: {N} nodes, {data.num_edges} edges')
print(f'Node features shape: {data.x.shape}')
print(f'Labels (communities): {data.y}')

In [ ]:
# Train the scratch GCN on Karate Club
torch.manual_seed(42)

model_scratch = GCNScratch(
    in_features=data.num_features,
    hidden=16,
    out_features=dataset.num_classes
)
optimizer = torch.optim.Adam(model_scratch.parameters(), lr=0.01, weight_decay=5e-4)

# Karate Club has only 4 labeled nodes (1 per community) in the default split
train_mask = data.train_mask

def train_epoch(model, X, A_hat, y, mask):
    model.train()
    optimizer.zero_grad()
    out = model(X, A_hat)
    loss = F.cross_entropy(out[mask], y[mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def accuracy(model, X, A_hat, y, mask):
    model.eval()
    with torch.no_grad():
        out = model(X, A_hat)
        pred = out.argmax(dim=1)
        return (pred[mask] == y[mask]).float().mean().item()

X = data.x
y = data.y

losses = []
for epoch in range(201):
    loss = train_epoch(model_scratch, X, A_hat, y, train_mask)
    losses.append(loss)
    if epoch % 50 == 0:
        acc = accuracy(model_scratch, X, A_hat, y, train_mask)
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | Train Acc: {acc:.4f}')

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GCN-from-scratch training loss')
plt.tight_layout()
plt.show()

---
## 4. PyG's GCNConv — The Production Way

The scratch implementation was for understanding. In practice:
- It doesn't scale (materializing the full N×N adjacency matrix is expensive)
- PyG's `GCNConv` uses sparse message passing — it only touches existing edges
- Same math, but O(E) memory instead of O(N²)

PyG's key design: layers take `(x, edge_index)` not `(x, A)`. The heavy lifting happens in `MessagePassing.propagate()`.

In [ ]:
# ============================================================
# GCN using PyG's GCNConv
# ============================================================

class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return x
    
    def embed(self, x, edge_index):
        """Return intermediate (hidden) embeddings — useful for visualization."""
        with torch.no_grad():
            x = F.relu(self.conv1(x, edge_index))
        return x


torch.manual_seed(42)
model_pyg = GCN(
    in_channels=data.num_features,
    hidden_channels=16,
    out_channels=dataset.num_classes
)
optimizer_pyg = torch.optim.Adam(model_pyg.parameters(), lr=0.01, weight_decay=5e-4)

print(model_pyg)
print(f'\nTotal params: {sum(p.numel() for p in model_pyg.parameters())}')

In [ ]:
def train_pyg(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test_pyg(model, data):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    return (pred == data.y).float().mean().item()

losses_pyg = []
for epoch in range(201):
    loss = train_pyg(model_pyg, data, optimizer_pyg)
    losses_pyg.append(loss)
    if epoch % 50 == 0:
        acc = test_pyg(model_pyg, data)
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | Full-graph Acc: {acc:.4f}')

In [ ]:
# Visualize learned node embeddings
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

model_pyg.eval()
embeddings = model_pyg.embed(data.x, data.edge_index).numpy()

if embeddings.shape[1] > 2:
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)
else:
    embeddings_2d = embeddings

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw graph
nx_g = to_networkx(data, to_undirected=True)
colors = plt.cm.tab10(data.y.numpy())
pos = nx.spring_layout(nx_g, seed=7)
nx.draw(nx_g, pos, ax=axes[0], node_color=colors, node_size=200,
        edge_color='lightgray', with_labels=True, font_size=7)
axes[0].set_title('Karate Club — true communities')

# Embeddings
for cls in data.y.unique():
    mask = data.y == cls
    axes[1].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   label=f'Community {cls.item()}', s=80)
for i, (x_e, y_e) in enumerate(embeddings_2d):
    axes[1].annotate(str(i), (x_e, y_e), fontsize=7, ha='center', va='bottom')
axes[1].set_title('Learned node embeddings (PCA)')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## 5. Inside MessagePassing — How PyG Really Works

All PyG layers inherit from `MessagePassing`. Under the hood:

```python
# Conceptually what propagate() does:
def propagate(self, edge_index, x):
    row, col = edge_index  # row=src, col=dst (or vice versa, check flow)
    # 1. message(): compute per-edge messages
    msg = self.message(x[col])  # features of source nodes
    # 2. aggregate(): sum/mean/max messages at destination
    agg = scatter(msg, row, dim=0, reduce='add')
    # 3. update(): final transformation
    return self.update(agg)
```

To write a custom GNN, subclass `MessagePassing` and override these methods.

In [ ]:
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

class MyGCNConv(MessagePassing):
    """Custom GCN layer subclassing MessagePassing."""
    
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')  # 'add' aggregation = sum
        self.lin = nn.Linear(in_channels, out_channels, bias=False)
        nn.init.xavier_uniform_(self.lin.weight)
    
    def forward(self, x, edge_index):
        # Step 1: add self-loops
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        # Step 2: linear transform
        x = self.lin(x)
        # Step 3: compute normalization
        row, col = edge_index
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
        # Step 4: propagate messages (calls message() then aggregate() then update())
        return self.propagate(edge_index, x=x, norm=norm)
    
    def message(self, x_j, norm):
        # x_j: features of source node j for each edge
        return norm.view(-1, 1) * x_j

# Test it
torch.manual_seed(0)
my_conv = MyGCNConv(data.num_features, 8)
out = my_conv(data.x, data.edge_index)
print(f'MyGCNConv output shape: {out.shape}')
print('Custom GCN layer works!')

---
# MINI-PROJECT 2: Implement Your Own GCN Layer

**Task:** Implement a variant of the GCN layer called **"mean aggregation GCN"** (like GraphSAGE's mean aggregator but in GCN style), train it on the Karate Club, and compare it to standard GCN.

**Your variant:** instead of symmetric normalization $D^{-1/2} A D^{-1/2}$, use simple mean aggregation: $\hat{A} = D^{-1} A$ (divide each row by its degree — average neighbor features).

**Requirements:**
1. Implement `MeanGCNLayer` as a subclass of `MessagePassing` (or use matrix ops)
2. Build a 2-layer model using your layer
3. Train on Karate Club for 200 epochs
4. Plot training loss
5. Compare final accuracy vs. the standard GCN above
6. Answer: which aggregation works better and why?

**Hint:** For mean aggregation, the `degree` normalization changes from `deg^{-0.5} * deg^{-0.5}` to just `deg^{-1}` (only applied once, at the destination node).

In [ ]:
# MINI-PROJECT SKELETON
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree

class MeanGCNConv(MessagePassing):
    """GCN with mean (D^{-1} A) normalization instead of symmetric."""
    
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')   # still sum — normalization handles the mean
        # TODO: define a linear layer
        pass
    
    def forward(self, x, edge_index):
        # TODO 1: add self-loops to edge_index
        
        # TODO 2: linear transform x
        
        # TODO 3: compute D^{-1} normalization (not symmetric)
        # Hint: norm[i->j] = 1 / degree(j)
        
        # TODO 4: call self.propagate(edge_index, x=x, norm=norm)
        pass
    
    def message(self, x_j, norm):
        # TODO: apply norm to source features x_j
        pass


class MeanGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # TODO: two MeanGCNConv layers
        pass
    
    def forward(self, x, edge_index):
        # TODO: conv1 -> relu -> dropout -> conv2
        pass


# Training loop
# TODO: train MeanGCN on data (same Karate Club data as above)
# Compare loss curves and final accuracy with model_pyg

# Uncomment to train:
# torch.manual_seed(42)
# model_mean = MeanGCN(data.num_features, 16, dataset.num_classes)
# optimizer_mean = torch.optim.Adam(model_mean.parameters(), lr=0.01)
# ...

print('Fill in the TODOs above and run!')

### Hints

<details>
<summary>Hint 1 — D^{-1} normalization</summary>

```python
row, col = edge_index
deg = degree(col, x.size(0), dtype=x.dtype)  # destination degrees
norm = 1.0 / deg[col]  # normalize by destination degree
```
</details>

<details>
<summary>Hint 2 — message()</summary>

```python
def message(self, x_j, norm):
    return norm.view(-1, 1) * x_j
```
</details>

<details>
<summary>Hint 3 — Plotting comparison</summary>

```python
plt.plot(losses_standard, label='Symmetric GCN')
plt.plot(losses_mean, label='Mean GCN')
plt.legend()
```
</details>

---
## What's Next?

In **Notebook 3**, we scale up to the Cora citation network (~2700 nodes), build a proper train/val/test pipeline, and dig into overfitting, dropout, and embedding visualization.